# 本次只运行末尾新增的两格

保留前面的历史记录。不要点“全部运行”，无需重复clone仓库、安装micromamba、CUDA或训练依赖。

跳到文末“当前实验：负残差强度”。第一格挂载Drive并上传 glsd_server_ready.zip，第二格运行SAMMLV新审计和搜索。
固定a0=2、rho=2，内层联合选择lambda_minus和tau；输出保存到Drive的GLSD_NEGATIVE_STRENGTH目录。


# ME-TST / GLSD 环境精简版

本 notebook 只保留原 `GLSD+ME-TST.ipynb` 中**最终成功建立运行环境所需的步骤**，
删除前面反复失败的 Python 3.10 / requirements / conda 尝试，以及错误的 dlib patch。

用途：
- 恢复 ME-TST 原工程；
- 建立原实验最终使用的 Python 3.8 + PyTorch 2.1.0 环境；
- 安装原 notebook 最终成功的依赖版本；
- 验证 `Utils`、`mamba_ssm`、`causal_conv1d` 可导入。

**不会运行 backbone、不会生成 response、不会修改 Drive 中 locked evidence。**

建议：Colab 运行时选择 GPU，然后从上到下依次执行。


## 1. 挂载 Google Drive


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("DRIVE_MOUNT = PASS")


## 2. 干净恢复 ME-TST 仓库

这里会删除 `/content/ME-TST` 的**临时 Colab 目录**后重新 clone。
不会删除 Google Drive 中的任何文件。
这样可以避免之前残留的不完整仓库导致 `Utils` 缺失。


In [ ]:
%cd /content
!rm -rf /content/ME-TST
!git clone https://github.com/zizheng-guo/ME-TST.git /content/ME-TST

from pathlib import Path

repo = Path("/content/ME-TST")
assert (repo / "Utils").exists(), "ME-TST/Utils not found after clean clone"
assert (repo / "Utils/mean_average_precision_str").exists(), "mean_average_precision_str folder missing"

print("METST_REPO_CLONE = PASS")
print("repo =", repo)


## 3. 安装 micromamba


In [ ]:
%cd /content
!rm -rf /content/bin
!wget -qO- https://micro.mamba.pm/api/micromamba/linux-64/latest | tar -xvj

from pathlib import Path
mamba = Path("/content/bin/micromamba")
assert mamba.exists(), "micromamba binary missing"
print("MICROMAMBA_INSTALL = PASS")
print("micromamba =", mamba)


## 4. 创建最终成功的 ME-TST 环境

对应原 notebook 最终采用的环境：
- Python 3.8
- PyTorch 2.1.0
- torchvision 0.16.0
- torchaudio 2.1.0
- CUDA 12.1


In [ ]:
!/content/bin/micromamba env remove -n ME-TST -y || true

!/content/bin/micromamba create -y -n ME-TST   -c pytorch -c nvidia -c conda-forge   python=3.8   pytorch=2.1.0   torchvision=0.16.0   torchaudio=2.1.0   pytorch-cuda=12.1   pip

!/content/bin/micromamba run -n ME-TST python --version


## 5. 验证 PyTorch / CUDA


In [ ]:
!/content/bin/micromamba run -n ME-TST python -c "import torch; print('torch =', torch.__version__); print('torch cuda =', torch.version.cuda); print('cuda available =', torch.cuda.is_available())"


## 6. 安装原 notebook 最终成功的基础依赖


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install   natsort==7.1.1   einops==0.8.0   tqdm==4.66.4   pandas==1.1.3   scikit-learn==0.23.2   scipy==1.5.4   xlrd==1.2.0   opencv-python-headless==4.5.1.48

!/content/bin/micromamba run -n ME-TST pip install --no-deps timm==1.0.7


## 7. 验证基础依赖与 ME-TST Utils


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST python -c "import torch,natsort,einops,timm,pandas,sklearn,scipy,cv2; print('torch =',torch.__version__); print('cuda =',torch.cuda.is_available()); print('BASIC_OK')"

!/content/bin/micromamba run -n ME-TST python -c "from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d; print('METST_UTILS_IMPORT = PASS')"


## 8. 安装 causal-conv1d


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install     causal-conv1d==1.4.0     --no-build-isolation


## 9. 安装 mamba-ssm


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST pip install     mamba-ssm==1.2.0.post1     --no-build-isolation


## 10. 最终环境检查


In [ ]:
%cd /content/ME-TST

!/content/bin/micromamba run -n ME-TST python -c "import sys, torch, pandas, mamba_ssm, causal_conv1d; from Utils.mean_average_precision_str.mean_average_precision import MeanAveragePrecision2d; print('Python:', sys.version); print('torch:', torch.__version__); print('CUDA:', torch.cuda.is_available()); print('pandas:', pandas.__version__); print('mamba_ssm: OK'); print('causal_conv1d: OK'); print('Utils metric: OK'); print('ME_TST_ENVIRONMENT = PASS')"


## 下一步

当最后一格输出 `ME_TST_ENVIRONMENT = PASS` 后，环境恢复完成。

**不要重新生成 official response。**
后续只需要读取 Drive 中已经锁定的 official dumps / evidence，
再加入 official-response 的 H/G/L/G+L 消融与 paired bootstrap。


In [ ]:
from google.colab import drive, files
from pathlib import Path
import zipfile, io, subprocess, sys
drive.mount('/content/drive')
uploaded = files.upload()
name = 'metst_fusion_screening.zip'
assert name in uploaded, '请上传 metst_fusion_screening.zip'
task_dir = Path('/content/glsd_fusion_screening')
task_dir.mkdir(exist_ok=True)
allowed = {'run_fusion_screening.py', 'official_response_component_ablation.py', 'README.md', 'fusion_screening_colab.ipynb'}
with zipfile.ZipFile(io.BytesIO(uploaded[name])) as archive:
    assert set(archive.namelist()) == allowed, '脚本包内容与预期不一致'
    archive.extractall(task_dir)
print('脚本已准备:', task_dir)


In [ ]:
from pathlib import Path
import io
import zipfile

assert len(uploaded) == 1, "请一次只上传一个脚本 ZIP 包"

actual_name, zip_data = next(iter(uploaded.items()))
print("实际上传文件名：", actual_name)

task_dir = Path("/content/glsd_fusion_screening")
task_dir.mkdir(parents=True, exist_ok=True)

allowed = {
    "run_fusion_screening.py",
    "official_response_component_ablation.py",
    "README.md",
    "fusion_screening_colab.ipynb",
}

with zipfile.ZipFile(io.BytesIO(zip_data)) as archive:
    actual_files = set(archive.namelist())
    assert actual_files == allowed, (
        f"脚本包内容与预期不一致：{sorted(actual_files)}"
    )
    archive.extractall(task_dir)

print("脚本包解压成功：", task_dir)

In [ ]:
from google.colab import files
from pathlib import Path

uploaded_full = files.upload()

assert len(uploaded_full) == 1, "请一次只上传一个 Python 脚本"

actual_name, file_bytes = next(iter(uploaded_full.items()))
print("实际上传文件名：", actual_name)

assert actual_name.lower().endswith(".py"), (
    f"上传的不是 Python 脚本：{actual_name}"
)

target = task_dir / "run_full_fusion_tuning.py"
target.write_bytes(file_bytes)

print("完整调参脚本已准备好：", target)

In [ ]:
from datetime import datetime, timezone
from pathlib import Path
import os
import subprocess
import sys

full_script = task_dir / "run_full_fusion_tuning.py"
assert full_script.is_file(), f"找不到脚本：{full_script}"

run_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")

output_base = Path("/content/drive/MyDrive/GLSD_FULL_FUSION_TUNING")
output_base.mkdir(parents=True, exist_ok=True)

screening_output = output_base / ("full_" + run_tag)
screening_log = output_base / ("full_" + run_tag + ".log")

bootstrap = r"""
import runpy
import sys
import pandas as pd

if not hasattr(pd.DataFrame, "append"):
    def dataframe_append(
        self,
        other,
        ignore_index=False,
        verify_integrity=False,
        sort=False,
    ):
        return pd.concat(
            [self, other],
            ignore_index=ignore_index,
            verify_integrity=verify_integrity,
            sort=sort,
        )

    pd.DataFrame.append = dataframe_append

sys.argv = sys.argv[1:]
runpy.run_path(sys.argv[0], run_name="__main__")
"""

command = [
    sys.executable,
    "-u",
    "-c",
    bootstrap,
    str(full_script),
    "--setting",
    "sammlv",
    "--output",
    str(screening_output),
]

env = dict(os.environ)
env["PYTHONPATH"] = os.pathsep.join(
    dict.fromkeys(
        [
            os.getcwd(),
            str(task_dir),
            *[p for p in sys.path if p],
            env.get("PYTHONPATH", ""),
        ]
    )
)

print("开始 SAMMLV 完整调参")
print("脚本：", full_script)
print("输出目录：", screening_output)
print("日志：", screening_log)

with screening_log.open("w", encoding="utf-8") as log:
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        env=env,
    )

    for line in process.stdout:
        print(line, end="")
        log.write(line)
        log.flush()

    returncode = process.wait()

assert returncode == 0, (
    f"实验失败，returncode={returncode}，请查看日志：{screening_log}"
)

print("SAMMLV_FULL_TUNING_FINISHED")
print("结果目录：", screening_output)

In [ ]:
from pathlib import Path
import ast
import json

# 打印被截图横向截断的具体参数
log_path = Path(screening_log)
for line in log_path.read_text(encoding="utf-8").splitlines():
    marker = "official decoder/count failure: "
    if marker in line and line.startswith("RuntimeError:"):
        details = json.loads(line.split(marker, 1)[1])
        print("出错参数：")
        print(json.dumps(details, ensure_ascii=False, indent=2))

# 核对当前 ME-TST 仓库的匹配函数
root = Path("/content/ME-TST/Utils")
found = False
for path in root.rglob("*.py"):
    if "mean_average_precision_str" not in str(path):
        continue
    source = path.read_text(encoding="utf-8")
    if "def check_box(" not in source:
        continue
    tree = ast.parse(source)
    for node in ast.walk(tree):
        if isinstance(node, ast.FunctionDef) and node.name == "check_box":
            found = True
            print("\n匹配函数文件：", path)
            print(ast.get_source_segment(source, node))

if not found:
    print("未找到 check_box，需要进一步确认实际评价器路径。")

In [ ]:
# Paste this entire file into ONE new cell in the prepared ME-TST+ notebook.
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import io
import os
import subprocess
import sys
import tempfile
import zipfile

# First fix and run SAMMLV. For CAS(ME)3 later set this to "casme3".
SETTING = "casme3"
PROBE_ONLY = False  # True runs the regression and Native without the full search.

assert Path("/content/drive/MyDrive").is_dir(), "请先挂载 Google Drive"
uploaded_oto = files.upload()  # Select metst_one_to_one_full_tuning.zip
assert len(uploaded_oto) == 1, "请一次只上传一个新 ZIP 包"
uploaded_name, package_bytes = next(iter(uploaded_oto.items()))
expected = {
    "run_one_to_one_full_tuning.py", "one_to_one_evaluator.py",
    "run_full_fusion_tuning.py", "run_fusion_screening.py",
    "official_response_component_ablation.py", "colab_one_to_one_entry.py",
    "README_ONE_TO_ONE.md",
}
oto_dir = Path(tempfile.mkdtemp(prefix="glsd_one_to_one_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
    assert len(archive.namelist()) == len(expected) and set(archive.namelist()) == expected, "ZIP 包内容不匹配"
    archive.extractall(oto_dir)

run_tag_oto = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
oto_root = Path("/content/drive/MyDrive/GLSD_ONE_TO_ONE_FULL_TUNING")
oto_root.mkdir(parents=True, exist_ok=True)
oto_output = oto_root / ("oto_" + run_tag_oto)
oto_log = oto_root / ("oto_" + run_tag_oto + ".log")
command_oto = [sys.executable, "-u", str(oto_dir / "colab_one_to_one_entry.py"),
               "--setting", SETTING, "--output", str(oto_output)]
if PROBE_ONLY:
    command_oto.append("--probe-only")
env_oto = dict(os.environ)
env_oto["PYTHONPATH"] = os.pathsep.join(dict.fromkeys([
    str(oto_dir), os.getcwd(), *[p for p in sys.path if p], env_oto.get("PYTHONPATH", "")
]))
print("上传文件：", uploaded_name)
print("运行目录：", oto_dir)
print("新结果目录：", oto_output)
print("日志：", oto_log)
with oto_log.open("x", encoding="utf-8") as log:
    process_oto = subprocess.Popen(command_oto, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, env=env_oto)
    try:
        for line in process_oto.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        code_oto = process_oto.wait()
    except BaseException:
        process_oto.terminate()
        try:
            process_oto.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process_oto.kill()
            process_oto.wait()
        raise
assert code_oto == 0, f"运行停止，请查看日志：{oto_log}"
assert (oto_output / "completion.json").is_file(), "未生成完成标志"
print("运行完成：", oto_output)


In [ ]:
# Paste this entire file into ONE new cell in the prepared ME-TST+ notebook.
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import io
import os
import subprocess
import sys
import tempfile
import zipfile

# First fix and run SAMMLV. For CAS(ME)3 later set this to "casme3".
SETTING = "sammlv"
THRESHOLD_GRID = "refined"  # All five methods: tau=0.05,...,0.95, step 0.05.
PROBE_ONLY = False  # True runs the regression and Native without the full search.

assert Path("/content/drive/MyDrive").is_dir(), "请先挂载 Google Drive"
uploaded_oto = files.upload()  # Select metst_one_to_one_refined_tuning.zip
assert len(uploaded_oto) == 1, "请一次只上传一个新 ZIP 包"
uploaded_name, package_bytes = next(iter(uploaded_oto.items()))
expected = {
    "run_one_to_one_full_tuning.py", "one_to_one_evaluator.py",
    "run_full_fusion_tuning.py", "run_fusion_screening.py",
    "official_response_component_ablation.py", "colab_one_to_one_entry.py",
    "README_ONE_TO_ONE.md",
}
oto_dir = Path(tempfile.mkdtemp(prefix="glsd_one_to_one_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(package_bytes)) as archive:
    assert len(archive.namelist()) == len(expected) and set(archive.namelist()) == expected, "ZIP 包内容不匹配"
    archive.extractall(oto_dir)

run_tag_oto = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
oto_root = Path("/content/drive/MyDrive/GLSD_ONE_TO_ONE_FULL_TUNING")
oto_root.mkdir(parents=True, exist_ok=True)
oto_output = oto_root / ("oto_refined_" + run_tag_oto)
oto_log = oto_root / ("oto_refined_" + run_tag_oto + ".log")
command_oto = [sys.executable, "-u", str(oto_dir / "colab_one_to_one_entry.py"),
               "--setting", SETTING, "--threshold-grid", THRESHOLD_GRID, "--output", str(oto_output)]
if PROBE_ONLY:
    command_oto.append("--probe-only")
env_oto = dict(os.environ)
env_oto["PYTHONPATH"] = os.pathsep.join(dict.fromkeys([
    str(oto_dir), os.getcwd(), *[p for p in sys.path if p], env_oto.get("PYTHONPATH", "")
]))
print("上传文件：", uploaded_name)
print("运行目录：", oto_dir)
print("新结果目录：", oto_output)
print("日志：", oto_log)
with oto_log.open("x", encoding="utf-8") as log:
    process_oto = subprocess.Popen(command_oto, stdout=subprocess.PIPE,
                                   stderr=subprocess.STDOUT, text=True, env=env_oto)
    try:
        for line in process_oto.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        code_oto = process_oto.wait()
    except BaseException:
        process_oto.terminate()
        try:
            process_oto.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process_oto.kill()
            process_oto.wait()
        raise
assert code_oto == 0, f"运行停止，请查看日志：{oto_log}"
assert (oto_output / "completion.json").is_file(), "未生成完成标志"
print("运行完成：", oto_output)



In [ ]:
# 在已经配置好的 ME-TST+ Colab notebook 中，新建一个代码 cell，粘贴本文件全文。
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

SETTING = "sammlv"

assert Path("/content/drive/MyDrive").is_dir(), "请先挂载 Google Drive"
uploaded_structure = files.upload()  # 选择 metst_one_to_one_structure_tuning.zip
assert len(uploaded_structure) == 1, "请一次只上传一个新 ZIP 包"
structure_name, structure_bytes = next(iter(uploaded_structure.items()))
structure_expected = {
    "run_one_to_one_full_tuning.py", "one_to_one_evaluator.py",
    "run_full_fusion_tuning.py", "run_fusion_screening.py",
    "official_response_component_ablation.py", "colab_one_to_one_entry.py",
    "structure_expansion.py", "refined_reference_sammlv.json", "README_STRUCTURE_TUNING_CN.md",
}
structure_dir = Path(tempfile.mkdtemp(prefix="glsd_structure_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(structure_bytes)) as archive:
    assert len(archive.namelist()) == len(structure_expected) and set(archive.namelist()) == structure_expected, "请上传新的结构扩展包，不能使用旧包"
    assert archive.testzip() is None, "ZIP 文件损坏"
    archive.extractall(structure_dir)

structure_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
structure_root = Path("/content/drive/MyDrive/GLSD_ONE_TO_ONE_FULL_TUNING")
structure_root.mkdir(parents=True, exist_ok=True)
structure_output = structure_root / ("oto_structure_" + structure_tag)
structure_log = structure_root / ("oto_structure_" + structure_tag + ".log")
structure_command = [sys.executable, "-u", str(structure_dir / "colab_one_to_one_entry.py"),
    "--setting", SETTING, "--threshold-grid", "refined", "--structure-grid", "expanded",
    "--output", str(structure_output)]
structure_env = dict(os.environ)
structure_env["PYTHONPATH"] = os.pathsep.join(dict.fromkeys([
    str(structure_dir), os.getcwd(), *[p for p in sys.path if p], structure_env.get("PYTHONPATH", "")
]))
print("上传文件：", structure_name)
print("新结果目录：", structure_output)
print("日志：", structure_log)
with structure_log.open("x", encoding="utf-8") as log:
    structure_process = subprocess.Popen(structure_command, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, env=structure_env)
    try:
        for line in structure_process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        structure_code = structure_process.wait()
    except BaseException:
        structure_process.terminate()
        try:
            structure_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            structure_process.kill()
            structure_process.wait()
        raise
assert structure_code == 0, f"运行停止，请查看日志：{structure_log}"
assert json.loads((structure_output / "completion.json").read_text())["completed"], "实验未完成"
structure_result_zip = shutil.make_archive(str(structure_output), "zip",
    root_dir=structure_output.parent, base_dir=structure_output.name)
print("实验运行完成；PASS 只表示代码执行成功，请结合结果判断方法效果。")
print("请下载并发回这个结果 ZIP：", structure_result_zip)


In [ ]:
# 在已经配置好的 ME-TST+ Colab notebook 中，新建一个代码 cell，粘贴本文件全文。
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile


assert Path("/content/drive/MyDrive").is_dir(), "请先挂载 Google Drive"
uploaded_structure = files.upload()  # 选择 metst_fusion_event_diagnostic.zip
assert len(uploaded_structure) == 1, "请一次只上传一个新 ZIP 包"
structure_name, structure_bytes = next(iter(uploaded_structure.items()))
structure_expected = {
    "run_fusion_event_diagnostic.py", "one_to_one_evaluator.py",
    "run_full_fusion_tuning.py", "run_fusion_screening.py",
    "official_response_component_ablation.py", "run_one_to_one_full_tuning.py",
    "structure_expansion.py", "structure_selected_replay_reference.json",
    "README_EVENT_DIAGNOSTIC_CN.md",
}
structure_dir = Path(tempfile.mkdtemp(prefix="glsd_event_diagnostic_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(structure_bytes)) as archive:
    assert len(archive.namelist()) == len(structure_expected) and set(archive.namelist()) == structure_expected, "请上传新的事件诊断包，不能使用旧搜索包"
    assert archive.testzip() is None, "ZIP 文件损坏"
    archive.extractall(structure_dir)

structure_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
structure_root = Path("/content/drive/MyDrive/GLSD_ONE_TO_ONE_FULL_TUNING")
structure_root.mkdir(parents=True, exist_ok=True)
structure_output = structure_root / ("oto_events_" + structure_tag)
structure_log = structure_root / ("oto_events_" + structure_tag + ".log")
structure_command = [sys.executable, "-u", str(structure_dir / "run_fusion_event_diagnostic.py"),
    "--output", str(structure_output)]
structure_env = dict(os.environ)
structure_env["PYTHONPATH"] = os.pathsep.join(dict.fromkeys([
    str(structure_dir), os.getcwd(), *[p for p in sys.path if p], structure_env.get("PYTHONPATH", "")
]))
print("上传文件：", structure_name)
print("新结果目录：", structure_output)
print("日志：", structure_log)
with structure_log.open("x", encoding="utf-8") as log:
    structure_process = subprocess.Popen(structure_command, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, env=structure_env)
    try:
        for line in structure_process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        structure_code = structure_process.wait()
    except BaseException:
        structure_process.terminate()
        try:
            structure_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            structure_process.kill()
            structure_process.wait()
        raise
assert structure_code == 0, f"运行停止，请查看日志：{structure_log}"
assert json.loads((structure_output / "completion.json").read_text())["completed"], "实验未完成"
structure_result_zip = shutil.make_archive(str(structure_output), "zip",
    root_dir=structure_output.parent, base_dir=structure_output.name)
print("诊断回放完成，已核对原预测和计数。请发回结果 ZIP 分析互补性。")
print("请下载并发回这个结果 ZIP：", structure_result_zip)


In [ ]:
# 在已经配置好的 ME-TST+ Colab notebook 中，新建一个代码 cell，粘贴本文件全文。
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile


assert Path("/content/drive/MyDrive").is_dir(), "请先挂载 Google Drive"
uploaded_structure = files.upload()  # 选择 metst_fixed_beta_replay.zip
assert len(uploaded_structure) == 1, "请一次只上传一个新 ZIP 包"
structure_name, structure_bytes = next(iter(uploaded_structure.items()))
structure_expected = {
    "run_fixed_beta_replay.py", "run_fusion_event_diagnostic.py", "one_to_one_evaluator.py",
    "run_full_fusion_tuning.py", "run_fusion_screening.py",
    "official_response_component_ablation.py", "run_one_to_one_full_tuning.py",
    "structure_expansion.py", "structure_selected_replay_reference.json",
    "fixed_beta_search_input.npz", "README_FIXED_BETA_CN.md",
}
structure_dir = Path(tempfile.mkdtemp(prefix="glsd_fixed_beta_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(structure_bytes)) as archive:
    assert len(archive.namelist()) == len(structure_expected) and set(archive.namelist()) == structure_expected, "请上传新的固定 beta 回放包，不能使用旧搜索包"
    assert archive.testzip() is None, "ZIP 文件损坏"
    archive.extractall(structure_dir)

structure_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
structure_root = Path("/content/drive/MyDrive/GLSD_ONE_TO_ONE_FULL_TUNING")
structure_root.mkdir(parents=True, exist_ok=True)
structure_output = structure_root / ("oto_fixed_beta_" + structure_tag)
structure_log = structure_root / ("oto_fixed_beta_" + structure_tag + ".log")
structure_command = [sys.executable, "-u", str(structure_dir / "run_fixed_beta_replay.py"),
    "--output", str(structure_output)]
structure_env = dict(os.environ)
structure_env["PYTHONPATH"] = os.pathsep.join(dict.fromkeys([
    str(structure_dir), os.getcwd(), *[p for p in sys.path if p], structure_env.get("PYTHONPATH", "")
]))
print("上传文件：", structure_name)
print("新结果目录：", structure_output)
print("日志：", structure_log)
with structure_log.open("x", encoding="utf-8") as log:
    structure_process = subprocess.Popen(structure_command, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, env=structure_env)
    try:
        for line in structure_process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        structure_code = structure_process.wait()
    except BaseException:
        structure_process.terminate()
        try:
            structure_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            structure_process.kill()
            structure_process.wait()
        raise
assert structure_code == 0, f"运行停止，请查看日志：{structure_log}"
assert json.loads((structure_output / "completion.json").read_text())["completed"], "实验未完成"
structure_result_zip = shutil.make_archive(str(structure_output), "zip",
    root_dir=structure_output.parent, base_dir=structure_output.name)
print("固定 beta 的 full 对照完成。请发回结果 ZIP；PASS 只表示执行与核验完成。")
print("请下载并发回这个结果 ZIP：", structure_result_zip)


In [ ]:
# 在已经配置好的 ME-TST+ Colab notebook 中，新建一个代码 cell，粘贴本文件全文。
from google.colab import files
from pathlib import Path
from datetime import datetime, timezone
import io
import json
import os
import shutil
import subprocess
import sys
import tempfile
import zipfile

SETTING = "casme3"

assert Path("/content/drive/MyDrive").is_dir(), "请先挂载 Google Drive"
uploaded_structure = files.upload()  # 选择 metst_casme3_fixed_max.zip
assert len(uploaded_structure) == 1, "请一次只上传一个新 ZIP 包"
structure_name, structure_bytes = next(iter(uploaded_structure.items()))
structure_expected = {
    "run_one_to_one_full_tuning.py", "one_to_one_evaluator.py",
    "run_full_fusion_tuning.py", "run_fusion_screening.py",
    "official_response_component_ablation.py", "colab_one_to_one_entry.py",
    "structure_expansion.py", "refined_reference_sammlv.json", "README_CASME3_FIXED_MAX_CN.md",
}
structure_dir = Path(tempfile.mkdtemp(prefix="glsd_casme3_", dir="/content"))
with zipfile.ZipFile(io.BytesIO(structure_bytes)) as archive:
    assert len(archive.namelist()) == len(structure_expected) and set(archive.namelist()) == structure_expected, "请上传新的结构扩展包，不能使用旧包"
    assert archive.testzip() is None, "ZIP 文件损坏"
    archive.extractall(structure_dir)

structure_tag = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
structure_root = Path("/content/drive/MyDrive/GLSD_ONE_TO_ONE_FULL_TUNING")
structure_root.mkdir(parents=True, exist_ok=True)
structure_output = structure_root / ("oto_casme3_max_" + structure_tag)
structure_log = structure_root / ("oto_casme3_max_" + structure_tag + ".log")
structure_command = [sys.executable, "-u", str(structure_dir / "colab_one_to_one_entry.py"),
    "--setting", SETTING, "--threshold-grid", "refined", "--structure-grid", "expanded", "--beta-grid", "fixed_max",
    "--output", str(structure_output)]
structure_env = dict(os.environ)
structure_env["PYTHONPATH"] = os.pathsep.join(dict.fromkeys([
    str(structure_dir), os.getcwd(), *[p for p in sys.path if p], structure_env.get("PYTHONPATH", "")
]))
print("上传文件：", structure_name)
print("新结果目录：", structure_output)
print("日志：", structure_log)
with structure_log.open("x", encoding="utf-8") as log:
    structure_process = subprocess.Popen(structure_command, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, env=structure_env)
    try:
        for line in structure_process.stdout:
            print(line, end="")
            log.write(line)
            log.flush()
        structure_code = structure_process.wait()
    except BaseException:
        structure_process.terminate()
        try:
            structure_process.wait(timeout=10)
        except subprocess.TimeoutExpired:
            structure_process.kill()
            structure_process.wait()
        raise
assert structure_code == 0, f"运行停止，请查看日志：{structure_log}"
assert json.loads((structure_output / "completion.json").read_text())["completed"], "实验未完成"
structure_result_zip = shutil.make_archive(str(structure_output), "zip",
    root_dir=structure_output.parent, base_dir=structure_output.name)
print("CAS(ME)3 固定 beta=1 验证运行完成；PASS 只表示执行成功，请结合结果判断方法效果。")
print("请下载并发回这个结果 ZIP：", structure_result_zip)


# 当前实验：负残差强度

只运行下面两格。需要 glsd_server_ready.zip。它与服务器使用同一个已验证算法包，名称无需改动。
不要运行旧的CAS(ME)3、fixed_beta、p8或环境重建cell。


CELL 1：挂载Drive，上传已验证的 glsd_server_ready.zip


In [ ]:
from google.colab import drive, files
from pathlib import Path
import io, json, os, subprocess, sys, tempfile, zipfile

drive.mount('/content/drive')
if sys.version_info < (3, 10):
    raise RuntimeError('请使用Colab默认Python内核，不要使用micromamba的Python 3.8')
NEG_DUMP = Path('/content/drive/MyDrive/ME-TST_OFFICIAL_DUMP/SAMMLV_method1_strategy1')
if len(list(NEG_DUMP.glob('subject_*.pkl'))) != 29:
    raise RuntimeError(f'需要29份SAMMLV响应，请检查Drive路径：{NEG_DUMP}')

uploaded_negative = files.upload()  # 只选择 glsd_server_ready.zip；重名后的(1)后缀不影响
if len(uploaded_negative) != 1:
    raise RuntimeError('请只上传一个实验ZIP')
negative_bytes = next(iter(uploaded_negative.values()))
negative_unpack = Path(tempfile.mkdtemp(prefix='glsd_negative_', dir='/content'))
with zipfile.ZipFile(io.BytesIO(negative_bytes)) as archive:
    for name in archive.namelist():
        path = Path(name)
        if path.is_absolute() or '..' in path.parts:
            raise RuntimeError('压缩包包含不支持的路径')
    if 'glsd_server_ready/run_standalone.py' not in archive.namelist():
        raise RuntimeError('请上传 glsd_server_ready.zip，不是之前的metst_asymmetric_residual.zip')
    archive.extractall(negative_unpack)
NEG_CODE = negative_unpack / 'glsd_server_ready'
check = subprocess.run([sys.executable, '-c', 'import numpy,pandas,scipy,sklearn'],
                       capture_output=True, text=True)
if check.returncode:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r',
                    str(NEG_CODE / 'requirements.txt')], check=True)
print('COLAB_NEGATIVE_SETUP = PASS')
print('Python =', sys.executable, sys.version.split()[0])
print('响应目录 =', NEG_DUMP)




CELL 2：新审计 + 负残差强度搜索，保存Drive并下载结果


In [ ]:
from datetime import datetime, timezone

# 首次运行保持None；中断续跑时填入上次打印的OUTPUT完整路径。
NEG_RESUME = None
negative_base = Path('/content/drive/MyDrive/GLSD_NEGATIVE_STRENGTH')
negative_base.mkdir(parents=True, exist_ok=True)
stamp = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
NEG_OUTPUT = Path(NEG_RESUME) if NEG_RESUME else negative_base / ('sammlv_' + stamp)
NEG_LOG = Path(str(NEG_OUTPUT) + '.log')
command = [sys.executable, '-u', str(NEG_CODE / 'run_standalone.py'),
           '--dump', str(NEG_DUMP), '--output', str(NEG_OUTPUT)]
if NEG_RESUME:
    command.append('--resume')
negative_env = os.environ.copy()
negative_env['PYTHONUNBUFFERED'] = '1'
# 子进程使用包内作者运行时，避免此前notebook导入或补丁残留。
negative_env.pop('PYTHONPATH', None)
print('OUTPUT =', NEG_OUTPUT, flush=True)
print('LOG =', NEG_LOG, flush=True)
with NEG_LOG.open('a', encoding='utf-8') as log:
    process_negative = subprocess.Popen(command, cwd=NEG_CODE, env=negative_env,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    try:
        for line in process_negative.stdout:
            print(line, end='', flush=True)
            log.write(line)
            log.flush()
        negative_exit = process_negative.wait()
    except KeyboardInterrupt:
        process_negative.terminate()
        try:
            process_negative.wait(timeout=10)
        except subprocess.TimeoutExpired:
            process_negative.kill()
            process_negative.wait()
        print('已停止子进程。续跑时将NEG_RESUME设为：', str(NEG_OUTPUT))
        raise

NEG_ZIP = Path(str(NEG_OUTPUT) + '.zip')
with zipfile.ZipFile(NEG_ZIP, 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(NEG_OUTPUT.rglob('*')):
        if path.is_file():
            archive.write(path, str(Path(NEG_OUTPUT.name) / path.relative_to(NEG_OUTPUT)))
    archive.write(NEG_LOG, NEG_OUTPUT.name + '/colab.log')
print('结果ZIP =', NEG_ZIP)
if negative_exit == 0:
    completed = json.loads((NEG_OUTPUT / 'completion.json').read_text())
    if not completed.get('completed'):
        raise RuntimeError('进程结束，但没有完成标记')
    import pandas as pd
    from IPython.display import display
    display(pd.read_csv(NEG_OUTPUT / 'summary_full.csv'))
    print('NEGATIVE_STRENGTH = PASS（执行完成，不代表方法有效）')
files.download(str(NEG_ZIP))
if negative_exit != 0:
    raise RuntimeError(f'运行失败，退出码={negative_exit}。错误日志已保存并打包：{NEG_ZIP}')
